# 02 · Depth regression

Analytic Pearson residuals — the SenePy input

**In** — `{dataset}_preprocessed.h5ad` (module 01)
**Out** — `{dataset}_pearson.h5ad`
- `.X` = Pearson residuals
- `layers['counts']` = raw counts
- `layers['lognorm']` = log-normalized

Senescence scores correlate with sequencing depth in brain snRNA-seq. Analytic
Pearson residuals model expected expression per gene under a negative-binomial
GLM with library size as covariate, so a residual is expression *beyond what
depth predicts*.

**This matrix feeds SenePy scoring and nothing else.** DEG uses `counts`; module
scoring, cell cycle, visualization and variability all use `lognorm`.

Lause et al. 2021, *Genome Biology* · Hafemeister & Satija 2019, *Genome Biology*

## Config

In [ ]:
# ===========================================================================
# CONFIG
# ===========================================================================
import os
from pathlib import Path

DATASET = 'psychad_aging'    # psychad_aging | psychad_ad | psychencode | mathys

DATA_ROOT = Path(os.environ.get('SENESCENCE_DATA', 'data'))

INPUT_FILE  = DATA_ROOT / 'processed' / f'{DATASET}_preprocessed.h5ad'
OUTPUT_FILE = DATA_ROOT / 'processed' / f'{DATASET}_pearson.h5ad'
FIGURES_DIR = DATA_ROOT / 'figures' / '02_depth_regression' / DATASET
RESULTS_DIR = DATA_ROOT / 'results' / '02_depth_regression' / DATASET
for d in (OUTPUT_FILE.parent, FIGURES_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

CLIP_VALUE   = 30         # residual clip, SCTransform v2 convention
N_DIAG_CELLS = 5_000      # cells sampled for depth diagnostics
N_DIAG_GENES = 100        # genes sampled for the per-gene diagnostic
STORE_DTYPE  = 'float32'  # residuals are dense; float64 doubles the file for no gain
SEED         = 0

print(f"  dataset : {DATASET}")
print(f"  in      : {INPUT_FILE}")
print(f"  out     : {OUTPUT_FILE}")
print(f"  clip    : +/-{CLIP_VALUE}   dtype: {STORE_DTYPE}")
if not INPUT_FILE.exists():
    print("\n  WARNING input not found - run module 01 first")

## Setup

In [ ]:
import time, platform
import scanpy as sc
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
import warnings; warnings.filterwarnings('ignore')

np.random.seed(SEED)
plt.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 300, 'font.size': 10,
    'axes.labelsize': 10, 'axes.titlesize': 11, 'legend.fontsize': 9,
    'font.family': 'sans-serif', 'axes.linewidth': 1.0,
    'axes.grid': False, 'pdf.fonttype': 42,
})

assert tuple(int(x) for x in sc.__version__.split('.')[:2]) >= (1, 9), \
    f"scanpy >= 1.9 required for Pearson residuals (have {sc.__version__})"

(RESULTS_DIR / 'run_info.txt').write_text(
    f"module  : 02_depth_regression\ndataset : {DATASET}\n"
    f"clip    : {CLIP_VALUE}\nseed    : {SEED}\n"
    f"python  : {platform.python_version()}\nscanpy  : {sc.__version__}\n")

print(f"scanpy {sc.__version__} - numpy {np.__version__}")

## Load

**Why.** Residuals are computed from raw counts, so `layers['counts']` must exist
and hold integers. `.X` should be log-normalized on arrival - if module 01 saved
something else, the residual computation would run on the wrong matrix and
produce numbers that look plausible.

In [ ]:
t0 = time.time()
adata = sc.read_h5ad(INPUT_FILE)
print(f"  {adata.n_obs:,} cells x {adata.n_vars:,} genes   ({time.time()-t0:.0f}s)")
print(f"  layers: {list(adata.layers.keys())}")

if 'counts' not in adata.layers:
    raise ValueError("layers['counts'] missing - module 01 must save raw counts")

_x = adata.X
_xmin = float(_x.data.min() if sp.issparse(_x) else _x.min())
_xmax = float(_x.data.max() if sp.issparse(_x) else _x.max())
print(f"  .X range: [{_xmin:.2f}, {_xmax:.2f}]   (expect ~0-10, log-normalized)")
if _xmin < 0:
    raise ValueError(".X has negative values - already residuals? Module 02 expects log-norm.")

_c = adata.layers['counts']
_s = np.asarray(_c.data[:1000] if sp.issparse(_c) else np.asarray(_c).flat[:1000])
if not np.allclose(_s, np.round(_s)):
    raise ValueError("layers['counts'] is not integer - not raw counts")
print(f"  counts max: {float(_c.data.max() if sp.issparse(_c) else _c.max()):,.0f}  - integers OK")

## Baseline depth coupling

**Why.** Recorded before correction so the effect is measured rather than
assumed. Sampled rather than exhaustive - the correlation is stable at 5,000
cells and computing it on 500K is wasteful.

In [ ]:
idx = np.random.choice(adata.n_obs, min(N_DIAG_CELLS, adata.n_obs), replace=False)
total_counts = adata.obs['total_counts'].iloc[idx].values

def _rowmean(M):
    return np.asarray(M.mean(axis=1)).ravel() if sp.issparse(M) else np.asarray(M).mean(axis=1)

rho_pre = spearmanr(_rowmean(adata.X[idx]), total_counts).statistic
print(f"  Spearman rho (mean .X vs total_counts) = {rho_pre:+.4f}")
print(f"  {'strong' if abs(rho_pre) > 0.8 else 'moderate' if abs(rho_pre) > 0.5 else 'low'} "
      f"depth coupling in log-normalized expression")

## Prepare

**Why.** The residual computation takes raw counts as `.X`. The log-normalized
matrix is preserved to `layers['lognorm']` first - it is what module scoring,
cell cycle, visualization and variability all read. The stale `uns['log1p']` flag
is cleared so scanpy does not treat the counts as already logged.

In [ ]:
adata.layers['lognorm'] = adata.X.copy()
adata.X = adata.layers['counts'].copy()
adata.uns.pop('log1p', None)

print("  layers['lognorm'] <- previous .X (log-normalized)")
print("  .X                <- raw counts (input for residuals)")

## Compute residuals

**Why.** For each gene, `residual = (observed - expected) / sqrt(variance)`, with
expected and variance modelled from the cell's library size. A gene that is high
only because the cell was sequenced deeply gives a residual near zero; expression
exceeding what depth predicts gives a large one. Clipped at +/-30 so rare,
lowly-expressed genes do not dominate.

**Memory.** Residuals are dense - roughly `n_cells x n_genes x 8` bytes at
float64. For 500K x 17K that is ~68 GB resident. `STORE_DTYPE = float32` halves
what gets written without affecting any downstream use.

In [ ]:
t0 = time.time()
sc.experimental.pp.normalize_pearson_residuals(adata, clip=CLIP_VALUE)
print(f"  computed in {(time.time()-t0)/60:.1f} min")
print(f"  .X: shape={adata.X.shape} dtype={adata.X.dtype} sparse={sp.issparse(adata.X)}")

if STORE_DTYPE and str(adata.X.dtype) != STORE_DTYPE:
    adata.X = adata.X.astype(STORE_DTYPE)
    print(f"  cast to {STORE_DTYPE}  (halves file size, no precision cost downstream)")

## Verification — and what this does *not* test

Two numbers are reported. The overall correlation should move toward zero. The
per-gene correlation typically gets *worse* after residual transformation, and
that is expected rather than alarming: residuals are signed and dense where
log-counts are non-negative and sparse, so Spearman against depth behaves
differently. **Neither number is a pass/fail gate.**

The gate that matters is senescence-score-versus-depth, and it lives in **module
03** - because residuals never reach a per-gene analysis. Their only consumer is
SenePy. Both numbers are recorded here so the module's effect is on file; the
decision about depth is made downstream where the score exists.

In [ ]:
rho_post = spearmanr(_rowmean(adata.X[idx]), total_counts).statistic

_min = float(adata.X.min()); _max = float(adata.X.max())
print(f"  overall rho   : {rho_pre:+.4f}  ->  {rho_post:+.4f}")
print(f"  .X range      : [{_min:.2f}, {_max:.2f}]   negatives present: {_min < 0}")
print(f"  clipped +/-{CLIP_VALUE}  : {_max <= CLIP_VALUE + 1e-6 and _min >= -CLIP_VALUE - 1e-6}")

gidx = np.random.choice(adata.n_vars, min(N_DIAG_GENES, adata.n_vars), replace=False)
tc = adata.obs['total_counts'].values
def _col(M, g):
    return np.asarray(M[:, g].todense()).ravel() if sp.issparse(M) else np.asarray(M[:, g]).ravel()
r_res = np.array([spearmanr(_col(adata.X, g), tc).statistic for g in gidx])
r_log = np.array([spearmanr(_col(adata.layers['lognorm'], g), tc).statistic for g in gidx])

print(f"\n  per-gene |rho|   lognorm {np.mean(np.abs(r_log)):.3f}   "
      f"residual {np.mean(np.abs(r_res)):.3f}   (n={len(gidx)} genes)")
print("  informational - not a gate; see the Why above")

pd.DataFrame({'rho_lognorm': r_log, 'rho_residual': r_res}
             ).to_csv(RESULTS_DIR / 'depth_coupling_per_gene.csv', index=False)
pd.DataFrame([{'dataset': DATASET, 'rho_pre': rho_pre, 'rho_post': rho_post,
               'clip': CLIP_VALUE}]).to_csv(RESULTS_DIR / 'depth_coupling.csv', index=False)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
ax[0].scatter(total_counts, _rowmean(adata.layers['lognorm'][idx]),
              s=2, alpha=.2, c='#4E79A7', rasterized=True)
ax[0].set(xlabel='total UMI', ylabel='mean expression',
          title=f'log-normalized  rho={rho_pre:.3f}')
ax[1].scatter(total_counts, _rowmean(adata.X[idx]),
              s=2, alpha=.2, c='#E15759', rasterized=True)
ax[1].set(xlabel='total UMI', ylabel='mean residual',
          title=f'Pearson residuals  rho={rho_post:.3f}')
ax[2].hist(r_log, bins=30, alpha=.6, color='#4E79A7', label='log-norm')
ax[2].hist(r_res, bins=30, alpha=.6, color='#E15759', label='residual')
ax[2].axvline(0, color='k', ls='--', lw=.8)
ax[2].set(xlabel='per-gene rho vs UMI', ylabel='genes'); ax[2].legend(frameon=False)
for a in ax:
    for s_ in ('top', 'right'): a.spines[s_].set_visible(False)
fig.suptitle(f'{DATASET} - depth coupling before and after', y=1.03)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'depth_coupling.pdf', bbox_inches='tight')
plt.show()

## Save

**Why.** Three layers, one contract:

`.X` — Pearson residuals — SenePy only
`layers['counts']` — raw counts — DEG
`layers['lognorm']` — log-normalized — everything else

This file is large and fully re-derivable from `counts` in minutes. Treat it as
an intermediate, not an archive.

In [ ]:
t0 = time.time()
adata.write_h5ad(OUTPUT_FILE)
gb = OUTPUT_FILE.stat().st_size / 1e9
print(f"  saved {OUTPUT_FILE.name}  ({gb:.1f} GB, {time.time()-t0:.0f}s)")
if gb > 20:
    print(f"  NOTE {gb:.0f} GB - dense residuals. Re-derivable; delete after module 03.")

## Gate

In [ ]:
def _gate(label, ok, detail=''):
    print(f"  [{'OK  ' if ok else 'FAIL'}] {label}{'  - ' + detail if detail else ''}")
    if not ok:
        raise AssertionError(f"GATE FAILED: {label}. {detail}")
    return ok

for _L in ('counts', 'lognorm'):
    _gate(f"layers['{_L}'] present", _L in adata.layers)
_gate(".X is residuals (has negatives)", float(adata.X.min()) < 0,
      f"min={float(adata.X.min()):.2f}")
_gate(f".X clipped to +/-{CLIP_VALUE}", float(adata.X.max()) <= CLIP_VALUE + 1e-6,
      f"max={float(adata.X.max()):.2f}")
_gate("shape non-degenerate", adata.n_obs > 0 and adata.n_vars > 0,
      f"{adata.n_obs:,} x {adata.n_vars:,}")

print(f"\n  depth coupling recorded: {rho_pre:+.4f} -> {rho_post:+.4f}")
print("  -> ready for module 03 - the score-vs-depth gate is there, not here")